## Load Document

In [1]:
import os
import asyncio
from pathlib import Path
import json
from dotenv import load_dotenv
from langchain_core.documents import Document
from typing import List
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

In [2]:
path = Path("../data/contracts")
pdf_paths = list(path.glob("*.pdf"))
processed_pdf = Path("../data/processed_contracts")
pdf_paths

[WindowsPath('../data/contracts/800263424202323950PMSLA-3208053.pdf'),
 WindowsPath('../data/contracts/OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).pdf')]

In [4]:
async def main():
    if not pdf_paths:
        raise ValueError("No PDF files found.")
    
    client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

    for pdf_path in pdf_paths:
        print(f"Processing {pdf_path.name}")

        with open(pdf_path, "rb") as f:
            file_obj = await client.files.create(file=f, purpose="parse")

        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="agentic_plus",
            version="latest",
            output_options={
                "markdown": {
                    "tables": {
                        "output_tables_as_markdown": True
                    }
                }
            },
            expand=["markdown"],
        )

        print(f"Result type: {type(result)}")
        print(f"Markdown: {result.markdown}")
        print(f"Result keys: {result.__dict__.keys()}")

        if result.markdown is None or result.markdown.pages is None:
            print(f"Warning: No markdown returned for {pdf_path.name}, trying text fallback...")
            
            markdown_result = await client.parsing.get_job_markdown(job_id=result.id)
            full_text = markdown_result if isinstance(markdown_result, str) else str(markdown_result)
        else:
            full_text = "\n\n".join(page.markdown for page in result.markdown.pages)

        output_file = processed_pdf / f"{pdf_path.stem}_parsed.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(full_text)

        print(f"Saved: {output_file}")

await main()

Processing 800263424202323950PMSLA-3208053.pdf
Result type: <class 'llama_cloud.types.parsing_get_response.ParsingGetResponse'>
Markdown: Markdown(pages=[MarkdownPageMarkdownResultPage(markdown='# SERVICE LEVEL AGREEMENT (SLA)\n\nMazagon Dock Shipbuilders Limited invites on-line competitive bids in **TWO BID SYSTEM** (Part-I Techno Commercial Bid and Part-II Price Bid), from reputed Bidders / Vendors, on www.gem.gov.in, for the Work/Services as detailed in this document:\n\n## 1. SUBJECT:\nBRC for hiring of services for grinding jobs along with grinding machines, grinding wheels and skilled workers for Project P17A at Yard Berths, Module Shop in-house and Project P15B Yard Berths.\n\n## 2. SCOPE OF WORK: As per Annexure-A\n* 2.1. The detailed Scope of Work (SoW) is attached herewith as Annexure-A.\n* 2.2. SLA envisages parallel order on 60:40 basis (value basis).\n* 2.3. Prices are to be quoted online.\n* 2.4. The total quantity indicated in rate sheet is tentative and may vary dependi

In [4]:
from pydantic import BaseModel, Field
from typing import Optional


class SupplierInfo(BaseModel):
    service_provider_name: Optional[str] = None
    client_name: Optional[str] = None
    agreement_date: Optional[str] = None
    contract_duration: Optional[str] = None
    model_config = {"extra": "forbid"}

class UptimeCommitments(BaseModel):
    guaranteed_uptime_percent: Optional[float] = None
    measurement_period: Optional[str] = None
    excluded_downtime: Optional[str] = None
    model_config = {"extra": "forbid"}

class PenaltyClause(BaseModel):
    clause_id: Optional[str] = None
    trigger_condition: Optional[str] = None
    penalty_type: Optional[str] = None
    penalty_amount: Optional[str] = None
    model_config = {"extra": "forbid"}

class TerminationClauses(BaseModel):
    termination_for_cause: Optional[str] = None
    notice_period_days: Optional[float] = None
    termination_for_convenience: Optional[str] = None
    model_config = {"extra": "forbid"}

class ForceMajeure(BaseModel):
    clause_id: Optional[str] = None
    covered_events: Optional[str] = None
    notification_requirement: Optional[str] = None
    model_config = {"extra": "forbid"}

class DisputeResolution(BaseModel):
    governing_law: Optional[str] = None
    resolution_mechanism: Optional[str] = None
    model_config = {"extra": "forbid"}

class SLADocument(BaseModel):
    supplier_info: Optional[SupplierInfo] = None
    uptime_commitments: Optional[UptimeCommitments] = None
    penalty_clauses: Optional[list[PenaltyClause]] = None
    termination_clauses: Optional[TerminationClauses] = None
    force_majeure: Optional[ForceMajeure] = None
    dispute_resolution: Optional[DisputeResolution] = None
    model_config = {"extra": "forbid"}

In [5]:
schema = SLADocument.model_json_schema()

from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

In [9]:
for pdf in pdf_paths:
    print(f"Processing: {pdf.name}")
    with open(pdf, "rb") as f:
        file_obj = client.files.create(file=(pdf.name, f, "application/pdf"), purpose="extract")

    result = client.extraction.extract(
        file_id=file_obj.id,
        data_schema=schema,
        config={},
    )

    # Debug: see what actually came back
    print("Raw result:", result.data)

    sla = SLADocument(**result.data)

    output_path = processed_pdf / f"{pdf.stem}.json"
    output_path.write_text(sla.model_dump_json(indent=2, exclude_none=True), encoding="utf-8")
    print(f"{pdf.name} → {output_path}")

    # Safe access
    if sla.supplier_info:
        print(f"  Provider : {sla.supplier_info.service_provider_name}")
    if sla.uptime_commitments:
        print(f"  Uptime   : {sla.uptime_commitments.guaranteed_uptime_percent}%")
    else:
        print("uptime_commitments not found in document")

Processing: 800263424202323950PMSLA-3208053.pdf
Raw result: {'supplier_info': {'service_provider_name': 'Mazagon Dock Shipbuilders Limited (MDL)', 'client_name': 'Mazagon Dock Shipbuilders Limited', 'agreement_date': None, 'contract_duration': 'two years from the date of placement of Order'}, 'uptime_commitments': None, 'penalty_clauses': None, 'termination_clauses': {'termination_for_cause': 'If the equipment / article / service or any portion thereof be not delivered/ performed by the scheduled delivery date/ period, any stoppage or discontinuation of ordered supply / awarded contract without written consent by Purchaser or not meeting the required quality standards the Purchaser shall be at liberty, without prejudice to the right of the Purchaser to recover Liquidated Damages / penalty as provided for in these conditions or to any other remedy for breach of contract, to terminate the contract either wholly or to the extent of such default.', 'notice_period_days': None, 'termination_

## Chunking

In [3]:
from langchain_community.document_loaders import TextLoader

processed_pdf = Path("../data/processed_contracts")
processed_pdf_path = list(processed_pdf.glob("*.txt"))
documents = []

for path in processed_pdf_path:
    loader = TextLoader(file_path=path, encoding='utf-8')
    docs = loader.load()
    documents.extend(docs)

print(documents)

e:\SentryChain-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '..\\data\\processed_contracts\\800263424202323950PMSLA-3208053_parsed.txt'}, page_content='# SERVICE LEVEL AGREEMENT (SLA)\n\nMazagon Dock Shipbuilders Limited invites on-line competitive bids in **TWO BID SYSTEM** (Part-I Techno Commercial Bid and Part-II Price Bid), from reputed Bidders / Vendors, on www.gem.gov.in, for the Work/Services as detailed in this document:\n\n## 1. SUBJECT:\nBRC for hiring of services for grinding jobs along with grinding machines, grinding wheels and skilled workers for Project P17A at Yard Berths, Module Shop in-house and Project P15B Yard Berths.\n\n## 2. SCOPE OF WORK: As per Annexure-A\n* 2.1. The detailed Scope of Work (SoW) is attached herewith as Annexure-A.\n* 2.2. SLA envisages parallel order on 60:40 basis (value basis).\n* 2.3. Prices are to be quoted online.\n* 2.4. The total quantity indicated in rate sheet is tentative and may vary depending upon the progress of work on the yards. Considering the priority, time

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 2000, chunk_overlap = 500):
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\nClause", "\nSection", " ", ""],
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Splitting {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content {split_docs[0].page_content[:100]}...")

    return split_docs

### Metadata:
* supplier
* section_id
* category(force majeure, penalty, termination)

In [5]:
processed_json_contracts = Path("../data/processed_contracts")
processed_json_contracts_paths = list(processed_json_contracts.glob("*.json"))
processed_json_contracts_paths

[WindowsPath('../data/processed_contracts/800263424202323950PMSLA-3208053.json'),
 WindowsPath('../data/processed_contracts/OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).json')]

In [23]:
def load_metadata(processed_pdf_path: list, processed_json_contracts_paths : list[Path]) -> list:
    docs = []
    supplier_lookup = {}

    for json_path in processed_json_contracts_paths:
        with open(json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)
        supplier_name = data.get("supplier_info", {}).get("service_provider_name", "Unknown")
        supplier_lookup[json_path.stem] = supplier_name

    for file_path in processed_pdf_path:
        with open(file_path, 'r', encoding = 'utf-8') as f:
            content = f.read()

        clean_stem = file_path.stem.replace("_parsed", "")

        doc = Document(
            page_content=content,
            metadata = {
                "supplier_id": file_path.stem,
                "supplier_name": supplier_lookup.get(clean_stem, "unknown"),
                "doc_type": "SLA",
                "source": str(file_path)
            }
        )
        docs.append(doc)

    return docs

documents = load_metadata(processed_pdf_path=processed_pdf_path, processed_json_contracts_paths=processed_json_contracts_paths)

chunks = split_docs(documents=documents)
chunks

Splitting 2 documents into 441 chunks
Content # SERVICE LEVEL AGREEMENT (SLA)

Mazagon Dock Shipbuilders Limited invites on-line competitive bids ...


[Document(metadata={'supplier_id': '800263424202323950PMSLA-3208053_parsed', 'supplier_name': 'Mazagon Dock Shipbuilders Limited (MDL)', 'doc_type': 'SLA', 'source': '..\\data\\processed_contracts\\800263424202323950PMSLA-3208053_parsed.txt'}, page_content='# SERVICE LEVEL AGREEMENT (SLA)\n\nMazagon Dock Shipbuilders Limited invites on-line competitive bids in **TWO BID SYSTEM** (Part-I Techno Commercial Bid and Part-II Price Bid), from reputed Bidders / Vendors, on www.gem.gov.in, for the Work/Services as detailed in this document:\n\n## 1. SUBJECT:\nBRC for hiring of services for grinding jobs along with grinding machines, grinding wheels and skilled workers for Project P17A at Yard Berths, Module Shop in-house and Project P15B Yard Berths.\n\n## 2. SCOPE OF WORK: As per Annexure-A\n* 2.1. The detailed Scope of Work (SoW) is attached herewith as Annexure-A.\n* 2.2. SLA envisages parallel order on 60:40 basis (value basis).\n* 2.3. Prices are to be quoted online.\n* 2.4. The total qua

In [24]:
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.prompts import PromptTemplate

In [25]:
from sentence_transformers import SentenceTransformer

texts = [chunk.page_content for chunk in chunks]

In [26]:
class EmbeddingManager:
    def __init__(self, model_name: str = "BAAI/bge-m3" ):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        print(f"Loading model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


    def generate_embeddings(self, texts: List[str]):
        embeddings = self.model.encode(texts, normalize_embeddings=True)
        if len(texts) == 1:
            return embeddings[0].astype(float).tolist()  
        return embeddings.astype(float).tolist()    

embedding_manager = EmbeddingManager()

Loading model: BAAI/bge-m3
Model loaded successfully. Embedding dimension: 1024


In [ ]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name= "sentry-chain-ai"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Manual index '{index_name}' created.")

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD")
)

dense_index = pc.Index(index_name)
for i, chunk in enumerate(chunks):
    const_id = f"{chunk.metadata['supplier']}#{i}"

    # push to neo4j
    graph.query("""
    MERGE (s:Supplier {name: $supplier})
    MERGE (c:Contract {name: 'SLA_2024'})
    MERGE (ch:Chunk {vector_id: $v_id})
    MERGE (s)-[:HAS_CONTRACT]->(c)
    MERGE (c)-[:HAS_DATA]->(ch)
    SET ch.text_preview = $text
    """, params={
        "supplier": chunk.metadata['supplier'],
        'v_id': const_id,
        "text": chunk.page_content 
    })

    # push to pinecone
    vector_values = embedding_manager.generate_embeddings([chunk.page_content])
    dense_index.upsert(vectors=[{
        "id": const_id,
        "values": vector_values,
        "metadata": {**chunk.metadata, "text": chunk.page_content}
    }])

[#EB85]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-04d2c4a4-b4aa.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Transaction failed and will be retried in 1.0113398229655413s (Failed to read from defunct connection IPv4Address(('si-04d2c4a4-b4aa.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))
[#C6C9]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 2.2419641676165734s (Unable to retrieve routing information)


In [14]:
from langchain_groq import ChatGroq

groq_llm = ChatGroq(api_key=os.getenv('GROQ_API_KEY'), model="llama-3.3-70b-versatile", temperature=0.1)

In [15]:
def retriever(query:str, supplier_name: str):
    # Vector search
    query_embedding = embedding_manager.generate_embeddings([query])
    vector_db_result = dense_index.query(
        vector=query_embedding,
        top_k=5,
        include_metadata=True
    )

    # Graph search
    graph_context = graph.query("""
    MATCH(s:Supplier{name:$supplier_name})-[:HAS_CONTRACT]->(c)-[:HAS_DATA]->(ch)
    WHERE ch.vector_id IN $vector_ids  
    RETURN ch.vector_id as id, ch.text_preview as preview
    """, params={
        "supplier_name": supplier_name,  
        "vector_ids": [m.id for m in vector_db_result['matches']] 
    })

    return graph_context, vector_db_result

In [22]:
def extract_supplier_name(raw_text: str) -> str:
    prompt = """Extract the service provider or supplier company name from this SLA text. 
    Return ONLY the company name, nothing else.
    
    Text: {text}""".format(text=raw_text[:500])
    
    response = groq_llm.invoke(prompt)
    return response.content.strip()

In [ ]:
def rag_query(query:str, supplier_name:str, extract:bool = True):

    display_name = supplier_name
    if extract:
        raw_chunks = graph.query("""
                MATCH (s:Supplier {name: $supplier_name})-[:HAS_CONTRACT]->(c)-[:HAS_DATA]->(ch)
                RETURN ch.text_preview as text LIMIT 1
            """, params={"supplier_name": supplier_name})
            
        if raw_chunks:
            display_name = extract_supplier_name(raw_chunks[0]['text'])
            print(f"Extracted supplier name: {display_name}")

    graph_context, vector_result = retriever(query=query, supplier_name=supplier_name)

    vector_texts = [match['metadata']['text'] for match in vector_result['matches']]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))

    prompt = PromptTemplate(
        template="""You are a contract analyst. Answer the question based ONLY on the provided context.

            Context from {supplier_name}'s SLA:
            {context}

            Question: {question}
            Answer (cite specific clauses when possible):""",
                    input_variables=["supplier_name", "context", "question"]
        )

    formatted_prompt = prompt.format(
        supplier_name=display_name,
        context="\n\n".join(combined_contexts),
        question=query
    )

    response = groq_llm.invoke(formatted_prompt)

    return {
        "answer": response.content,
        "sources": [m['id'] for m in vector_result['matches'][:3]],
        "context_used": combined_contexts,
        "display_name": display_name
    }


result = rag_query(
    query="What are the penalty clauses for service downtime?",
    supplier_name="OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed"
)

print(result['answer'])
print(f"\nSources: {result['sources']}")
print(f"\n {result['display_name']}")

Extracted supplier name: Microsoft
The penalty clauses for service downtime are outlined in the section "System Maintenance & Support Services" (PART C). According to this section, the Service Provider shall guarantee a minimum of 99.5% uptime per month for the Digital modules and Learning Management System (LMS) for C-PEC.

In case of non-conformity with the requisite uptime, the Service Provider shall be liable to pay the following amounts as liquidated damages (as per the clause "Penalty for shortfall in Performance compliance level"):

| Sl. No | Shortfall in Performance | Penalty  |
| ------ | ------------------------ | -------- |
| 1      | >0.5% and <= 1%          | 50,000   |
| 2      | >1 % and <=3%            | 1,00,000 |
| 3      | >3% and <= 5%            | 1,50,000 |
| 4      | >5%                      | 2,00,000 |

These penalties are calculated on a monthly basis for the shortfall in performance compliance level. The total of such penalties shall not exceed the amount sp

In [ ]:
# allowed_nodes=["Supplier", "Service", "Penalty", "Clause", "Uptime"],
# allowed_relationships=["HAS_PENALTY", "DEFINES", "GUARANTEES", "APPLIES_TO"]

# llm_transformer = LLMGraphTransformer(llm=groq_llm, allowed_nodes=allowed_nodes, allowed_relationships=allowed_relationships)
# graph_documents = await llm_transformer.aconvert_to_graph_documents(chunks)
# graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=True)

## Phase 2 News Fetching

In [ ]:
from langchain_tavily import TavilySearch, TavilyExtract

def fetch_news(supplier_name:str = result['display_name'], score_threshold:float = 0.5):
    tavily = TavilySearch(api_key=os.getenv('TAVILY_API_KEY'),
                        search_depth="advanced",
                        max_results=5)

    query = f"{supplier_name} supply chain disruption OR outage OR compliance risk OR SLA breach 2025"

    results = tavily.invoke(query)

    filtered_results = [r for r in results['results'] if r['score'] >= score_threshold]
    return filtered_results


In [34]:
supplier_news_result = fetch_news()

In [35]:
supplier_news_result

[{'url': 'https://greyhoundresearch.com/microsoft-365-outage-on-june-17-2025-what-happened/',
  'title': 'Microsoft 365 Outage on June 17, 2025: What Happened?',
  'content': 'operational cost of a control-plane disruption routinely exceeds $2 million per hour. This figure reflects not just lost transactions, but penalties from SLA breaches, delayed reconciliations, audit lapses, and executive decision-making paralysis. In several client environments, even a 60-minute outage during end-of-quarter processing has triggered contractual escalations and risk committee intervention. [...] part of the post-exercise remediation strategy, the CIO’s office prioritised proxy-based caching of identity sessions and invested in telemetry overlays to monitor orchestration drift. This type of scenario is increasingly informing business continuity investments across regulated industries in EMEA and APAC, where the cost of even a two-hour disruption extends beyond downtime into audit exposure, SLA breac

In [ ]:
def compare_sla(supplier_name : str, news_results : list):
    news_content = "".join([f"Source:{article['title']}\n{article['content']}" for article in news_results])

    query = "service outage uptime penalty SLA breach downtime"
    graph_context, vector_result = retriever(query=query , supplier_name=supplier_name)
    vector_texts = [match['metadata']['text'] for match in vector_result['matches']]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))
    prompt = f"""You are a contract risk analyst. 

        NEWS EVENTS:
        {news_content}

        SLA CLAUSES:
        {combined_contexts}

        Based on the news events above, analyze:
        1. Has an SLA violation likely occurred? (Yes/No/Unclear)
        2. Which specific clauses are triggered?
        3. What penalties or remedies apply?
        4. Overall risk severity? (Low/Medium/High/Critical)

        Be specific and cite clause numbers where possible."""

    response = groq_llm.invoke(prompt)
    return {
        "supplier_id": supplier_name,
        "verdict": response.content,
        "news_used": [a['title'] for a in news_results],
        "sla_clauses_matched": combined_contexts
    }